In [14]:
import os
from collections import Counter
import datetime as dt
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as ipw

from eval_genome import test_genome_mp
import eval_genome
eval_genome.METRIC = 'ttd'

In [19]:
def test_genome_against_n(genome: tuple[float, float, float, float], n_range, rng_seed=20, samples=100):
    results: list[dict] = []
    for n in n_range:
        stats, _rate = test_genome_mp(genome, n, rng_seed=rng_seed, trials=samples, tqdm_kwargs=dict(leave=True))
        print('n: ', n, sum(stats, Counter()))

        for stat in stats:
            results.append({"n": n, **stat})

    return results


out = ipw.Output()
out

Output()

In [20]:
n_range = [
    *range(1, 10),
    *range(10, 20, 2),
    *range(20, 100 + 1, 5),
]

with out:
    results = test_genome_against_n(
        (0.2, 0.2, 0.2, -0.2),
        n_range=n_range
    )

In [ ]:
df = pd.DataFrame.from_dict(results)
df

,n,Time to Detect,Time to Detect sub,Time to Capture sub,How many red on goal
0,1,1,-1,-1,1
1,1,1,-1,-1,1
2,1,1,-1,-1,1
3,1,1,-1,-1,1
4,1,1,-1,-1,1
...,...,...,...,...,...
3095,100,0,349,-1,0
3096,100,0,215,-1,0
3097,100,0,199,-1,0
3098,100,0,295,-1,0


In [ ]:
%matplotlib widget


def srate(df, name='Success'):
    x = (100 - df.groupby(['n'])['How many red on goal'].mean() * 100).round().rename(f'{name} Rate')
    return x.index, x


plt.plot(*srate(df), label='Something')
plt.legend()
plt.scatter(*srate(df))

dt_title = dt.datetime.now().strftime("%Y-%m-%d")
plt.title(dt_title, size='small', x=0.485)
plt.suptitle(f"{'Capture' if eval_genome.METRIC == 'ttc' else 'Detection'} Rate")
plt.xlabel('Number of Agents')
plt.ylabel('Success Rate (%)')
margin = 3
plt.ylim(0 - margin, 100 + 3 * margin)

for n, sr in zip(*srate(df)):
    plt.text(n, sr + 0.5, str(sr), ha="center", va="bottom")
